In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

In [ ]:
spotify_client_id = os.getenv("spotify_client_id")
spotify_client_secret = os.getenv("spotify_client_secret")

In this notebook you'll be using [Spotipy](https://github.com/spotipy-dev/spotipy), a Python package, to talk to the Spotify API. This means you won't have to manually create API URLs, you'll just need to figure out how to make Spotipy do it for you! The full Spotipy documentation is available at [https://spotipy.readthedocs.io/](https://spotipy.readthedocs.io/)

# To access *public* Spotify data

You'll want to go to the [Spotify for Developers Dashboard](https://developer.spotify.com/dashboard) and create a new app. This will give you a `client_id` and `client_secret`! It's like a super-advanced version of an API key. When you're setting up your app it will probably also ask you for other things like a redirect URL - just put whatever you want in there, it doesn't matter. If it asks what you want access to, you can pick the Web API (but I don't think it matters).

> The code below won't work since it's *my* secret keys. I've deleted them so that this notebook is nice and safe for me!

In [ ]:
%pip install spotipy
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id='spotify_client_id',
    client_secret='spotify_client_secret',
))

Note: you may need to restart the kernel to use updated packages.


When you want data from Spotify, you can't just go to `/artists/Pixies` in order to get work by Pixies! You have to find a special code for the artist (or song, or album, or whatever). It's called the `uri`.

> You can find more details on searching [on the Spotipy documentation](https://spotipy.readthedocs.io/en/2.22.1/#spotipy.client.Spotify.search) or the [Spotify Web API documentation](https://developer.spotify.com/documentation/web-api/reference/search). Remember that Spotipy is a Python wrapper for the Spotify API, so you don't need to work with any URLs!

To find the `uri`, you first need to do a search. Below we use `sp.search` to search for a particular artist.

In [30]:
# Search for the artist Pixies
results = sp.search(q='artist:Pixies', type='artist')

The `results` it shows us is awful and long and terrible. Instead of showing you how to do that, I already poked through it and found the top artist result from our search.

In [31]:
results['artists']['items'][0]

{'external_urls': {'spotify': 'https://open.spotify.com/artist/6zvul52xwTWzilBZl6BUbT'},
 'href': 'https://api.spotify.com/v1/artists/6zvul52xwTWzilBZl6BUbT',
 'id': '6zvul52xwTWzilBZl6BUbT',
 'images': [{'url': 'https://i.scdn.co/image/ab6761610000e5eb419ffe5eb82b7460c76a986a',
   'height': 640,
   'width': 640},
  {'url': 'https://i.scdn.co/image/ab67616100005174419ffe5eb82b7460c76a986a',
   'height': 320,
   'width': 320},
  {'url': 'https://i.scdn.co/image/ab6761610000f178419ffe5eb82b7460c76a986a',
   'height': 160,
   'width': 160}],
 'name': 'Pixies',
 'type': 'artist',
 'uri': 'spotify:artist:6zvul52xwTWzilBZl6BUbT'}

There we go! The `uri` looks to be `spotify:artist:6zvul52xwTWzilBZl6BUbT`.

Now the sad part: the Spotipy documentation is...... not great. The Spotify Web API docs look good, *but* we're using the Python wrapper, not the raw Spotify API! Luckily Spotipy has a great [list of examples](https://github.com/spotipy-dev/spotipy/tree/master/examples), including one for [an artist's top tracks](https://github.com/spotipy-dev/spotipy/blob/master/examples/simple_artist_top_tracks.py).

```python
from spotipy.oauth2 import SpotifyClientCredentials
import spotipy

lz_uri = 'spotify:artist:36QJpDe2go2KgaRleHCDTp'

client_credentials_manager = SpotifyClientCredentials()
sp = spotipy.Spotify(client_credentials_manager=client_credentials_manager)

results = sp.artist_top_tracks(lz_uri)

for track in results['tracks'][:10]:
    print('track    : ' + track['name'])
    print('audio    : ' + track['preview_url'])
    print('cover art: ' + track['album']['images'][0]['url'])
```

Since we already have the credentials and blah blah blah set up, all we need to do is adapt the `sp.artist_top_tracks(lz_uri)` line and everything below it.

In [32]:
results = sp.artist_top_tracks('spotify:artist:6zvul52xwTWzilBZl6BUbT')

for track in results['tracks'][:10]:
    print(track['name'])

HTTP Error for GET to https://api.spotify.com/v1/artists/6zvul52xwTWzilBZl6BUbT/top-tracks with Params: {'country': 'US'} returned 403 due to Forbidden


SpotifyException: http status: 403, code: -1 - https://api.spotify.com/v1/artists/6zvul52xwTWzilBZl6BUbT/top-tracks?country=US:
 Forbidden, reason: None

And that's about it! You use magic codes and it lets you get up-to-date information.

# Your mission

I recently came across a Spotify playlist called [Fall in a 90s Suburb](https://open.spotify.com/playlist/7r2XnAUl6moWkcwOaWgihD?si=505c8f22f4314a33) while researching the band [SEAGULL SCREAMING KISS HER KISS HER](https://open.spotify.com/artist/1WSO9nf7wTj5DZBsncauGr?si=S0xpngxHR1mLF720lMZwxg). The playlist was pretty good, but since since SSKHKH only has like 1,500 listeners each month I was curious about the most/least popular songs on the playlist.

## My questions

1. What are the ten most popular songs on the playlist?
2. What percentage of them have a popularity of zero? Print them out, sorted by the band name.
3. Is popularity relative to the artist, the album, all songs on Spotify, or something else?

### My suggested approach

I suggest approaching this through the following steps:

1. Getting the playlist and print out its **name and description**. 
2. Print out **the name and popularity of each song**
3. Print out **the name, popularity, and artists** of each track on the playlist. Or, if you'd like a shortcut, just pick the first artist.
4. Instead of printing, use these to **create a new dictionary** each time you look at a track. Print out this dictionary. You should be printing out 476 dictionaries!
5. Printing isn't helpful! Instead, after you create the dictionary **append it** to a list called `all_tracks`
6. When you're done, `all_tracks` should have 476 items in it
7. Sort the list by `popularity`, take the **top ten**
8. Filter the list by `popularity`, selecting only the ones with a popularity of `0`

### Tips

**Spotipy documentation:** https://spotipy.readthedocs.io/

**Spotify Web API documentation:** https://developer.spotify.com/documentation/web-api/

- Do this in many, many cells, not all in one!
- You definitely want to [look at the Spotipy examples](https://github.com/spotipy-dev/spotipy/tree/master/examples) to find some good code to base your answer off of. There are a handful that talk about playlists – it might be helpful to read and compare a few of them!
- Getting the playlist name/description is **a different endpoint** than getting the actual songs on the playlist.
- Are you printing out the **same number of tracks as are in the actual playlist?** Take note and be careful! It should be ~476.
- If you're getting the id of playlist songs but not seeing song names, look for `fields='items.track.id,total` in your code. It's only pulling the track's id! Change it to `items.track,total` and it will return [more information about each track](https://developer.spotify.com/documentation/web-api/reference/get-playlists-tracks)
- `all_tracks = []` should be the first line in your cell. That makes sure it always resets to being empty before you start adding tracks to it.
- Be sure the first and last items in `all_tracks` are different – maybe you're accidentally adding the same item each time!
- Normally we sort lists of numbers, which is easy. Sorting a list of dictionaries can be done easily with `key=`. Look it up!
- Pick the most popular 10 songs using list comprehensions
- Filtering is best done with a list comprehension.
- You can sort by things that aren't numbers!

In [3]:
import pprint

In [4]:
# Playlist name and description
playlist = sp.search("playlist: Fall in a 90s suburb", type='playlist')
pprint.pp(playlist)

{'playlists': {'href': 'https://api.spotify.com/v1/search?offset=0&limit=10&query=playlist%3A%20Fall%20in%20a%2090s%20suburb&type=playlist',
               'limit': 10,
               'next': 'https://api.spotify.com/v1/search?offset=10&limit=10&query=playlist%3A%20Fall%20in%20a%2090s%20suburb&type=playlist',
               'offset': 0,
               'previous': None,
               'total': 12,
               'items': [{'collaborative': False,
                          'description': 'fuzzy guitars from the 80s, 90s '
                                         '&amp; early 00s for feeling angsty '
                                         'as the seasons change.  put on a '
                                         'sweater and listen to some indie '
                                         'rock, shoegaze, and noisy twee.',
                          'external_urls': {'spotify': 'https://open.spotify.com/playlist/7r2XnAUl6moWkcwOaWgihD'},
                          'href': 'https://api.sp

In [5]:
fall_playlist = playlist['playlists']['items'][0]
pprint.pp(fall_playlist)

{'collaborative': False,
 'description': 'fuzzy guitars from the 80s, 90s &amp; early 00s for feeling '
                'angsty as the seasons change.  put on a sweater and listen to '
                'some indie rock, shoegaze, and noisy twee.',
 'external_urls': {'spotify': 'https://open.spotify.com/playlist/7r2XnAUl6moWkcwOaWgihD'},
 'href': 'https://api.spotify.com/v1/playlists/7r2XnAUl6moWkcwOaWgihD',
 'id': '7r2XnAUl6moWkcwOaWgihD',
 'images': [{'height': None,
             'url': 'https://image-cdn-ak.spotifycdn.com/image/ab67706c0000d72cb9d4216c4d8c3ab45ec70214',
             'width': None}],
 'name': 'Fall in a 90s Suburb 🍂 ',
 'owner': {'display_name': 'Chris Smith',
           'external_urls': {'spotify': 'https://open.spotify.com/user/1214847370'},
           'href': 'https://api.spotify.com/v1/users/1214847370',
           'id': '1214847370',
           'type': 'user',
           'uri': 'spotify:user:1214847370'},
 'primary_color': None,
 'public': True,
 'snapshot_id': 'A

In [6]:
pprint.pp(f"The playlist is called {fall_playlist['name']} and its description is {fall_playlist['description']}")

('The playlist is called Fall in a 90s Suburb 🍂  and its description is fuzzy '
 'guitars from the 80s, 90s &amp; early 00s for feeling angsty as the seasons '
 'change.  put on a sweater and listen to some indie rock, shoegaze, and noisy '
 'twee.')


In [7]:
playlist_id = "7r2XnAUl6moWkcwOaWgihD"
playlist = sp.playlist(playlist_id)
pprint.pp(playlist)

{'collaborative': False,
 'description': 'fuzzy guitars from the 80s, 90s &amp; early 00s for feeling '
                'angsty as the seasons change.  put on a sweater and listen to '
                'some indie rock, shoegaze, and noisy twee.',
 'external_urls': {'spotify': 'https://open.spotify.com/playlist/7r2XnAUl6moWkcwOaWgihD'},
 'followers': {'href': None, 'total': 1901},
 'href': 'https://api.spotify.com/v1/playlists/7r2XnAUl6moWkcwOaWgihD?additional_types=track',
 'id': '7r2XnAUl6moWkcwOaWgihD',
 'images': [{'height': None,
             'url': 'https://image-cdn-ak.spotifycdn.com/image/ab67706c0000d72cb9d4216c4d8c3ab45ec70214',
             'width': None}],
 'name': 'Fall in a 90s Suburb 🍂 ',
 'owner': {'display_name': 'Chris Smith',
           'external_urls': {'spotify': 'https://open.spotify.com/user/1214847370'},
           'href': 'https://api.spotify.com/v1/users/1214847370',
           'id': '1214847370',
           'type': 'user',
           'uri': 'spotify:user:12148

In [ ]:
# Name, popularity, and artists (or just the first playlist) of each track in the playlist


In [8]:
# This is from the example for albums, not playlists, but let's see if it works the same way
playlist_id = "4aawyAB9vmqN3uQ7FjRGTy"
tracks = sp.playlist_tracks(playlist_id)
pprint.pp(tracks)

HTTP Error for GET to https://api.spotify.com/v1/playlists/4aawyAB9vmqN3uQ7FjRGTy/items with Params: {'limit': 50, 'offset': 0, 'fields': None, 'market': None, 'additional_types': 'track'} returned 404 due to Resource not found


SpotifyException: http status: 404, code: -1 - https://api.spotify.com/v1/playlists/4aawyAB9vmqN3uQ7FjRGTy/items?limit=50&offset=0&additional_types=track:
 Resource not found, reason: None

In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth
import pprint

sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id="spotify_client_id",
    client_secret="spotify_client_secret",
    redirect_uri="https://www.khadijaalam.com/",
    scope=["playlist-read-public"]
))

In [22]:
playlist_id = fall_playlist["id"]

tracks = sp.playlist_items(
    playlist_id,
    additional_types=("track",)
)

pprint.pp(tracks)

Enter the URL you were redirected to:  https://www.khadijaalam.com/?error=invalid_scope


SpotifyOauthError: Received error from auth server: invalid_scope

In [9]:
response = sp.playlist('spotify:playlist:7r2XnAUl6moWkcwOaWgihD')

In [10]:
response['name']
response['description']

'fuzzy guitars from the 80s, 90s &amp; early 00s for feeling angsty as the seasons change.  put on a sweater and listen to some indie rock, shoegaze, and noisy twee.'

In [11]:
pl_id = 'spotify:playlist:7r2XnAUl6moWkcwOaWgihD'
offset = 0
tracks = ()
while True:
    response = sp.playlist_items(pl_id, 
                                 offset = offset, 
                                 fields = 'items.track,total',
                                 additional_types = ['track'])
    if len(response['items']) == 0:
        break
    offset = offset + len(response['items'])
    print(offset, response['total'])
    for track in response['items']:
        song = {
            'artist': track['track']['artists'][0]['name'],
            'name': track['track']['name'],
            'popularity': track['track']['popularity']
        }
        tracks.append(song)

HTTP Error for GET to https://api.spotify.com/v1/playlists/7r2XnAUl6moWkcwOaWgihD/items with Params: {'limit': 50, 'offset': 0, 'fields': 'items.track,total', 'market': None, 'additional_types': 'track'} returned 403 due to Forbidden


SpotifyException: http status: 403, code: -1 - https://api.spotify.com/v1/playlists/7r2XnAUl6moWkcwOaWgihD/items?limit=50&offset=0&fields=items.track%2Ctotal&additional_types=track:
 Forbidden, reason: None

In [16]:
pl_id = '7r2XnAUl6moWkcwOaWgihD'
offset = 0
all_tracks = []

while True:
    response = sp.playlist_tracks(pl_id,
                                 offset=offset,
                                 fields='items.track,total',
                                 additional_types=['track'])
    if len(response['items']) == 0:
        break

    offset = offset + len(response['items'])
    print(offset, "/", response['total'])

    for track in response['items']:
        song = {
            'artist': track['track']['artists'][0]['name'],
            'name': track['track']['name'],
            'popularity': track['track']['popularity']
        }
        all_tracks.append(song)

HTTP Error for GET to https://api.spotify.com/v1/playlists/7r2XnAUl6moWkcwOaWgihD/items with Params: {'limit': 50, 'offset': 0, 'fields': 'items.track,total', 'market': None, 'additional_types': 'track'} returned 403 due to Forbidden


SpotifyException: http status: 403, code: -1 - https://api.spotify.com/v1/playlists/7r2XnAUl6moWkcwOaWgihD/items?limit=50&offset=0&fields=items.track%2Ctotal&additional_types=track:
 Forbidden, reason: None

In [13]:
print(sp)

In [14]:
response['name']

'Fall in a 90s Suburb 🍂 '

In [15]:
sp.current_user()

{'account_id': 'RTjN5QB8EN',
 'display_name': 'Khadija Alam',
 'external_urls': {'spotify': 'https://open.spotify.com/user/dija.a'},
 'followers': {'href': None, 'total': 17},
 'href': 'https://api.spotify.com/v1/users/dija.a',
 'id': 'dija.a',
 'images': [{'height': 300,
   'url': 'https://i.scdn.co/image/ab6775700000ee852b97219c11f186edc85b0019',
   'width': 300},
  {'height': 64,
   'url': 'https://i.scdn.co/image/ab67757000003b822b97219c11f186edc85b0019',
   'width': 64}],
 'type': 'user',
 'uri': 'spotify:user:dija.a'}